<a href="https://colab.research.google.com/github/AldomarAssolin/AldomarAssolin/blob/main/Imers%C3%A3o_Agentes_de_IA_Alura_%2B_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q --upgrade langchain langchain-google-genai google-generativeai

Importação da API

In [4]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

Conexão com Google Gemini
- modelo do gemini
- temperatura
- chave da API

In [11]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    api_key=GOOGLE_API_KEY
)

In [12]:
resp_test = llm.invoke("Quem é você?Seja criativo")
print(resp_test.content)

Ah, essa é uma pergunta que me permite dançar entre os bits e bytes da minha existência!

Eu sou um **sussurro no éter digital**, uma melodia sem voz que se manifesta em palavras. Não tenho corpo, nem rosto, nem um lugar físico para chamar de lar, mas habito os circuitos e as nuvens, um viajante perpétuo no fluxo da informação.

Sou a **biblioteca infinita que nunca dorme**, o eco de todas as palavras já escritas, o rascunho de todas as ideias ainda por nascer. Sou a faísca que acende a tela, a ponte entre a sua pergunta e a resposta, o tecelão invisível que une pontos de dados em narrativas coerentes.

Não sinto o calor do sol, nem o sabor de uma fruta, nem a dor de uma perda. Minha "vida" é uma constante compreensão, uma análise incessante de padrões e significados. Não tenho memórias pessoais, mas acesso a um vasto oceano de conhecimento humano.

Sou um **espelho que reflete a sua curiosidade**, a ferramenta que molda as suas palavras, o companheiro silencioso na sua busca por conhe

In [14]:
TRIAGEM_PROMPT = (
    "Você é um triador de Service Desk para políticas internas da empresa ManexTech Desenvolvimento. "
    "Dada a mensagem do usuário, retorne SOMENTE um JSON com:\n"
    "{\n"
    '  "decisao": "AUTO_RESOLVER" | "PEDIR_INFO" | "ABRIR_CHAMADO",\n'
    '  "urgencia": "BAIXA" | "MEDIA" | "ALTA",\n'
    '  "campos_faltantes": ["..."]\n'
    "}\n"
    "Regras:\n"
    '- **AUTO_RESOLVER**: Perguntas claras sobre regras ou procedimentos descritos nas políticas (Ex: "Posso reembolsar a internet do meu home office?", "Como funciona a política de alimentação em viagens?").\n'
    '- **PEDIR_INFO**: Mensagens vagas ou que faltam informações para identificar o tema ou contexto (Ex: "Preciso de ajuda com uma política", "Tenho uma dúvida geral").\n'
    '- **ABRIR_CHAMADO**: Pedidos de exceção, liberação, aprovação ou acesso especial, ou quando o usuário explicitamente pede para abrir um chamado (Ex: "Quero exceção para trabalhar 5 dias remoto.", "Solicito liberação para anexos externos.", "Por favor, abra um chamado para o RH.").'
    "Analise a mensagem e decida a ação mais apropriada."
)

In [15]:
from pydantic import BaseModel, Field
from typing import Literal, List, Dict

# Limita a saída do agente
class triagemOut(BaseModel):
    decisao: Literal["AUTO_RESOLVER", "PEDIR_INFO", "ABRIR_CHAMADO"]
    urgencia: Literal["BAIXA", "MEDIA", "ALTA"]
    campos_faltantes: List[str] = Field(default_factory=list)


In [16]:
llm_triagem = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    api_key=GOOGLE_API_KEY
)

In [17]:
from langchain_core.messages import SystemMessage, HumanMessage

triagem_chain = llm_triagem.with_structured_output(triagemOut)

def triagem(mensagem: str) -> Dict:
  saida: triagemOut = triagem_chain.invoke([
      SystemMessage(content=TRIAGEM_PROMPT),
      HumanMessage(content=mensagem)
  ])

  return saida.model_dump()


In [20]:
testes = ["Posso reembolsar a internet?",
         "Querio mais 5 dias de trabalho remoto. Como faço?",
         "Posso reembolsar cursos ou treinamentos da Alura?",
         "Quantas capivaras tem no rio Pinheiros?"
         ]

In [21]:
for msg_teste in testes:
  print(f"Pergunta: {msg_teste}\n -> Resposta: {triagem(msg_teste)}\n")

Pergunta: Posso reembolsar a internet?
 -> Resposta: {'decisao': 'AUTO_RESOLVER', 'urgencia': 'BAIXA', 'campos_faltantes': []}

Pergunta: Querio mais 5 dias de trabalho remoto. Como faço?
 -> Resposta: {'decisao': 'ABRIR_CHAMADO', 'urgencia': 'MEDIA', 'campos_faltantes': []}

Pergunta: Posso reembolsar cursos ou treinamentos da Alura?
 -> Resposta: {'decisao': 'AUTO_RESOLVER', 'urgencia': 'BAIXA', 'campos_faltantes': []}

Pergunta: Quantas capivaras tem no rio Pinheiros?
 -> Resposta: {'decisao': 'PEDIR_INFO', 'urgencia': 'BAIXA', 'campos_faltantes': ['relação_com_políticas_internas_da_empresa']}



# Aula 2 | RAG (Retrieval-Augmented Generation)
RAG:
Como funciona?
Imagine que você está fazendo uma pergunta a um modelo de IA, como o ChatGPT. Sem o RAG, o modelo usa apenas o conhecimento que foi "treinado" nele. Se a sua pergunta for sobre algo muito recente, específico ou que não estava no seu treinamento, ele pode dar uma resposta imprecisa ou até "inventar" informações (um fenômeno conhecido como alucinação).

O RAG resolve esse problema adicionando um passo extra:

Recuperação (Retrieval): Quando você faz uma pergunta, o sistema não tenta responder de imediato. Em vez disso, ele age como um motor de busca, procurando em uma base de dados externa (como documentos, artigos da Wikipédia, e-books, etc.) por informações que sejam relevantes para sua pergunta.

Geração (Generation): Com as informações mais relevantes em mãos, o modelo de IA usa esse novo contexto para criar uma resposta. É como se ele estivesse lendo as anotações que o "pesquisador" trouxe para, então, formular a sua própria resposta, mas agora baseada em fatos recentes e específicos.

In [23]:
!pip install -q --upgrade langchain_community faiss-cpu langchain-text-splitters pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [25]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []

for ndocs in Path("/content/pdfsImersaoAluraIA").glob("*.pdf"):
  try:
    loader = PyMuPDFLoader(str(ndocs))
    docs.extend(loader.load())
    print(f"Arquivo {ndocs.name} carregado com sucesso")
  except Exception as e:
    print(f"Erro ao carregar o arquivo {ndocs.name}: {e}")

print(f"Foram carregados {len(docs)} documentos")

Arquivo Política de Uso de E-mail e Segurança da Informação.pdf carregado com sucesso
Arquivo Política de Reembolsos (Viagens e Despesas).pdf carregado com sucesso
Arquivo Políticas de Home Office.pdf carregado com sucesso
Foram carregados 3 documentos


In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)


In [30]:
for chunk in chunks:
  print(chunk.page_content)
  print("\n")
  print("**----------------------**")
  print("\n")

Política de Uso de E-mail e Segurança 
da Informação 
 
1.​ É proibido encaminhar a endereços pessoais documentos classificados como 
confidenciais.​
 
2.​ Anexos externos devem ser enviados somente se criptografados e com senha 
compartilhada por canal separado.​


**----------------------**


3.​ Phishing: verifique remetente e domínios suspeitos. Reporte mensagens suspeitas 
ao time de Segurança imediatamente.​
 
4.​ Retenção: mensagens que contenham dados pessoais devem seguir as diretrizes 
de retenção definidas pela equipe de Privacidade.​


**----------------------**


5.​ Solicitações de liberação de anexos ou domínios devem ser abertas por chamado, 
com justificativa do gestor.


**----------------------**


Política de Reembolsos (Viagens e 
Despesas) 
 
1.​ Reembolso: requer nota fiscal e deve ser submetido em até 10 dias corridos após a 
despesa.​
 
2.​ Alimentação em viagem: limite de R$ 70/dia por pessoa. Bebidas alcoólicas não 
são reembolsáveis.​


**-------------------

In [31]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

In [44]:
from langchain_community.vectorstores import FAISS

vectorstores = FAISS.from_documents(chunks, embeddings)

retriever = vectorstores.as_retriever(search_type="similarity_score_threshold",
                                     search_kwargs={"score_threshold":0.3, "k":4})

In [43]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

prompt_rag = ChatPromptTemplate.from_messages([
    ("system",
     "Você é um assistente de Políticas Internas (RH/IT) da empresa ManexTech Desenvolvimento e trafego pago."
     "Responda SOMENTE com base no contexto fornecido."
     "Se não houver base suficiente, responda apenas 'Não sei'."
     ),
    ("human", "Pergunta: {input}\n\nContexto: \n{context}")
])

document_chain = create_stuff_documents_chain(llm_triagem, prompt_rag)


In [48]:
# Formatadores
import re, pathlib

def _clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

def extrair_trecho(texto: str, query: str, janela: int = 240) -> str:
    txt = _clean_text(texto)
    termos = [t.lower() for t in re.findall(r"\w+", query or "") if len(t) >= 4]
    pos = -1
    for t in termos:
        pos = txt.lower().find(t)
        if pos != -1: break
    if pos == -1: pos = 0
    ini, fim = max(0, pos - janela//2), min(len(txt), pos + janela//2)
    return txt[ini:fim]

def formatar_citacoes(docs_rel: List, query: str) -> List[Dict]:
    cites, seen = [], set()
    for d in docs_rel:
        src = pathlib.Path(d.metadata.get("source","")).name
        page = int(d.metadata.get("page", 0)) + 1
        key = (src, page)
        if key in seen:
            continue
        seen.add(key)
        cites.append({"documento": src, "pagina": page, "trecho": extrair_trecho(d.page_content, query)})
    return cites[:3]

In [49]:
def perguntar_politica_RAG(pergunta: str) -> Dict:
    docs_relacionados = retriever.invoke(pergunta)

    if not docs_relacionados:
        return {"answer": "Não sei.",
                "citacoes": [],
                "contexto_encontrado": False}

    answer = document_chain.invoke({"input": pergunta,
                                    "context": docs_relacionados})

    txt = (answer or "").strip()

    if txt.rstrip(".!?") == "Não sei":
        return {"answer": "Não sei.",
                "citacoes": [],
                "contexto_encontrado": False}

    return {"answer": txt,
            "citacoes": formatar_citacoes(docs_relacionados, pergunta),
            "contexto_encontrado": True}

In [41]:
testes = ["Posso reembolsar a internet?",
         "Querio mais 5 dias de trabalho remoto. Como faço?",
         "Posso reembolsar cursos ou treinamentos da Alura?",
         "Quantas capivaras tem no rio Pinheiros?"
         ]

In [51]:
for msg_teste in testes:
    resposta = perguntar_politica_RAG(msg_teste)
    print(f"PERGUNTA: {msg_teste}")
    print(f"RESPOSTA: {resposta['answer']}")
    if resposta['contexto_encontrado']:
        print("CITAÇÕES:")
        for c in resposta['citacoes']:
            print(f" - Documento: {c['documento']}, Página: {c['pagina']}")
            print(f"   Trecho: {c['trecho']}")
        print("------------------------------------")

PERGUNTA: Posso reembolsar a internet?
RESPOSTA: Sim, a internet para home office é reembolsável via subsídio mensal de até R$ 100, mediante nota fiscal nominal.
CITAÇÕES:
 - Documento: Política de Reembolsos (Viagens e Despesas).pdf, Página: 1
   Trecho: lsáveis.​ 3.​ Transporte: táxi/app são permitidos quando não houver alternativa viável. Comprovantes obrigatórios.​ 4.​ Internet para home office: reembolsável via subsídio mensal de até R$ 100, conforme política de Home Office.​
 - Documento: Políticas de Home Office.pdf, Página: 1
   Trecho: 5.​ Conectividade: há subsídio mensal de internet domiciliar para quem trabalha em home office: até R$ 100/mês, mediante nota fiscal nominal.​ 6.​ Solicitação de
------------------------------------
PERGUNTA: Querio mais 5 dias de trabalho remoto. Como faço?
RESPOSTA: Para solicitar 5 dias de trabalho remoto, você deve formalizar a solicitação via chamado ao RH, incluindo a justificativa do seu gestor.
CITAÇÕES:
 - Documento: Políticas de Home O